In [ ]:
import numpy as np
from numpy.typing import NDArray
from typing import Optional
import pandas as pd
import matplotlib.pyplot as plt
from numba import jit

In [ ]:
ticks_dataset = pd.read_parquet("../data/ES_01_Calm_Aug2017.parquet", columns=["action", "ts_event", "price", "size"])
ticks_dataset.head()

In [ ]:
def data_cleaner_for_bars(ticks_dataset: pd.DataFrame) -> pd.DataFrame:
    # Filtering our dataset
    df_clean = ticks_dataset.loc[
        ticks_dataset["action"] == "T",
        ["ts_event", "price", "size"],].copy()
    
    # Deleting the indexes
    df_clean.reset_index(drop=True, inplace=True)
    return df_clean

In [ ]:
ticks_dataset = data_cleaner_for_bars(ticks_dataset)
ticks_dataset.head()

In [ ]:
def tick_bars_creator(ticks_dataset: pd.DataFrame, 
                      threshold: int = 1000) -> pd.DataFrame:
    
    n_bars = len(ticks_dataset) // threshold
    
    # Case not enough datas
    if n_bars == 0:
        bars = pd.DataFrame(columns=['open', 'high', 'low', 'close', 'vwap', 'volume', 'tick_count'])
        bars.index.name = "ts_event"
        return bars
    
    # Computing the number of ticks to keep
    nb_ticks_kept = n_bars * threshold
    
    # Creating the array versions of columns for computations
    arr_price = ticks_dataset["price"].values[:nb_ticks_kept]
    arr_size = ticks_dataset["size"].values[:nb_ticks_kept]
    arr_ts_event = ticks_dataset["ts_event"].values[:nb_ticks_kept]
    
    # Reshaping our arrays to only take into account the right prices
    price_matrix = arr_price.reshape(n_bars, threshold)
    size_matrix = arr_size.reshape(n_bars, threshold)
    ts_event_matrix = arr_ts_event.reshape(n_bars, threshold)
    
    # Compute volume and dollar values of ticks
    vol_sum = size_matrix.sum(axis=1)
    dollar = np.einsum("ij,ij->i", price_matrix, size_matrix)
    
    # Computing the tick bars
    bars = pd.DataFrame({
        'open': price_matrix[:, 0],
        'high': price_matrix.max(axis=1),
        'low': price_matrix.min(axis=1),
        'close': price_matrix[:, -1],
        'vwap': dollar / vol_sum,
        'volume': vol_sum,
        'tick_count': threshold
    }, index=ts_event_matrix[:, -1])
    bars.index.name = 'ts_event'
    
    return bars

In [ ]:
def volume_bars_creator(ticks_dataset: pd.DataFrame, 
                        threshold: int = 6000) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    # Vectorized Grouping
    data["cum_volume"] = data["size"].cumsum()
    data["bar_id"] = (data["cum_volume"] // threshold).astype(np.int32)
    
    # Computing bars by groups
    groups = data.groupby('bar_id')
    bars = groups.agg({
        'ts_event': 'last',
        'price': ['first', 'max', 'min', 'last'],
        'size': ['sum', 'count']
    })
    
    # Naming bars columns
    bars.columns = ['ts_event', 'open', 'high', 'low', 'close', 'volume', 'tick_count']
    
    # Computing vwap
    data["dollar_value"] = data['price'] * data['size']
    vwap_num = data.groupby('bar_id')['dollar_value'].sum()
    bars["vwap"] = vwap_num / bars['volume']
    
    # Setting index on bars
    bars.set_index(keys='ts_event', inplace=True)
    
    # If the last bar is icomplete delete it
    if not bars.empty:
        if bars.iloc[-1]['volume'] < 0.8 * threshold:
            bars = bars.iloc[:-1]
    
    
    return bars[['open', 'high', 'low', 'close', 'vwap', 'volume', 'tick_count']]

In [ ]:
def dollar_bars_creator(ticks_dataset: pd.DataFrame, 
                        threshold: float = 1.4e7) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["dollar_value"] = data["price"] * data["size"]
    data["dollar_value_cum"] = data["dollar_value"].cumsum()
    data["bar_id"] = (data["dollar_value_cum"] // threshold).astype(np.int32)
    
    bars = data.groupby(by="bar_id")
    bars = bars.agg({
        "ts_event": "last",
        "price": ['first', 'max', 'min', 'last'],
        "size": ['sum', 'count'],
        "dollar_value": ['sum']
    })
    bars.columns = ['ts_event', 'open', 'high', 'low', 'close', 'volume', 'tick_count', 'dollar_sum']
    
    # If the last bar is icomplete delete it
    if not bars.empty:
        if bars.iloc[-1]['dollar_sum'] < 0.8 * threshold:
            bars = bars.iloc[:-1]

    bars['vwap'] = bars['dollar_sum'] / bars['volume']
    bars.set_index(keys="ts_event", inplace=True)
    
    return bars[['open', 'high', 'low', 'close', 'vwap', 'volume', 'tick_count']]

In [ ]:
def tick_rule_creator(price_series: pd.Series) -> NDArray[np.floating]:
    # Compute price difference sign
    diff_sign: pd.Series = np.sign(price_series.diff())
    
    # Replace zeros by previous values
    diff_sign = diff_sign.replace(0, np.nan).ffill()
    diff_sign.iloc[0] = np.nan
    
    return diff_sign.values

In [ ]:
@jit(nopython=True)
def get_imbalance_bars_numba(tick_signs: np.ndarray, 
                             initial_T: float = 1000.,
                             initial_E_b: float = 0., 
                             min_bar_length: int = 10,
                             span: int = 1000) -> NDArray[np.int32]:
    
    # Length of our tick dataset
    T = len(tick_signs)
    
    # Initializing vector of zeros
    bar_indices = np.zeros(T, dtype=np.int32)
    bar_count = 0
    
    # Initializing theta
    theta = 0.
    alpha = 2. / (span + 1)
    
    # Initializing expectations
    E_T = initial_T
    E_b = initial_E_b
    
    # initializing index of the last bar tick
    last_idx = -1
    
    # Computing min_threshold
    min_expected_imbalance = np.maximum(np.abs(initial_E_b) * 1e-4, 1e-6)
    
    # Computing the bars indexes
    for i in range(T):
        theta += tick_signs[i]
        
        # Thresold computation
        expected_imbalance = np.maximum(np.abs(E_b), min_expected_imbalance)
        threshold = E_T * expected_imbalance
        
        if np.abs(theta) >= threshold and (i - last_idx) >= min_bar_length:
            # add the index
            bar_indices[bar_count] = i
            bar_count += 1
            
            # update the current mean
            current_T = i - last_idx
            current_b = theta / current_T
            
            # update the ewma
            E_T = (1 - alpha) * E_T + alpha * current_T
            E_b = (1 - alpha) * E_b + alpha * current_b
            
            # Reset
            theta = 0
            last_idx = i
            
    # return only the non empty values
    return bar_indices[:bar_count]

In [ ]:
def apply_imbalance_bars(df: pd.DataFrame, bar_indices: NDArray[np.int32]) -> pd.DataFrame:
    
    # Initialize imbalance bars by zeros
    bar_ids = np.zeros(len(df), dtype=np.int32)
    
    # The bar indices are the last index of the ticks in the bar 
    # so we make sure it is less than the lenght of our dataset 
    # and we add 1 because the next bar begin after the last index
    cut_locations = bar_indices[bar_indices < len(df) - 1] + 1
    
    # We put 1 at each new bar start
    bar_ids[cut_locations] = 1
    
    # With cumsum, we compute the bar id, given by the cumcum
    # indeed the cumsums are different for each bars
    df['bar_id'] = np.cumsum(bar_ids)
    
    # Compute dollar value of the tick
    df['dollar_value'] = df['price'] * df['size']
        
    bars = df.groupby('bar_id').agg({
        'ts_event': 'last',
        'price': ['first', 'max', 'min', 'last'],
        'size': ['sum', 'count'],
        'dollar_value': 'sum'
    })
    
    # 4. Nettoyage
    bars.columns = ['ts_event', 'open', 'high', 'low', 'close', 'volume', 'tick_count', 'dollar_sum']
    bars['vwap'] = bars['dollar_sum'] / bars['volume']
    bars.set_index('ts_event', inplace=True)
    
    return bars[['open', 'high', 'low', 'close', 'vwap', 'volume', 'tick_count']]

In [ ]:
def tick_imbalance_bar_creator(ticks_dataset: pd.DataFrame,
                               ticks_first_bar: int = 1000) -> pd.DataFrame:
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["tick_rule"] = tick_rule_creator(data["price"])
    data.dropna(subset='tick_rule', inplace=True)
    
    b = data["tick_rule"].values.astype(np.int8)
    initial_E_b = b[:ticks_first_bar].mean()
    
    bar_indices = get_imbalance_bars_numba(b, initial_E_b=initial_E_b, initial_T=ticks_first_bar)
    
    return apply_imbalance_bars(data, bar_indices)

In [ ]:
def volume_imbalance_bar_creator(ticks_dataset: pd.DataFrame, 
                                 ticks_first_bar: int = 1000) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["tick_rule"] = tick_rule_creator(data["price"])
    data.dropna(subset='tick_rule', inplace=True)
    
    bv = data["size"].values * data["tick_rule"].values.astype(np.float64)
    initial_bv = bv[:ticks_first_bar].mean()
    
    bar_indices = get_imbalance_bars_numba(bv, initial_T=ticks_first_bar, initial_E_b=initial_bv)
    
    return apply_imbalance_bars(data, bar_indices)

In [ ]:
def dollar_imbalance_bar_creator(ticks_dataset: pd.DataFrame, 
                                 ticks_first_bar: int = 1000) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["tick_rule"] = tick_rule_creator(data["price"])
    data.dropna(subset='tick_rule', inplace=True)
    
    bd = (data['price'] * data["size"] * data["tick_rule"]).values.astype(np.float64)
    initial_bv = bd[:ticks_first_bar].mean()
    
    bar_indices = get_imbalance_bars_numba(bd, initial_T=ticks_first_bar, initial_E_b=initial_bv)
    
    return apply_imbalance_bars(data, bar_indices)

In [ ]:
@jit(nopython=True)
def get_runs_bars_numba(tick_signs: np.ndarray, 
                        initial_T: float = 1000.,
                        initial_E_b_plus: float = 0.5,
                        initial_E_b_minus: float = 0.5, 
                        min_bar_length: int = 10,
                        span: int = 1000) -> NDArray[np.int32]:
    
    # Length of our tick dataset
    T = len(tick_signs)
    
    # Initializing vector of zeros
    bar_indices = np.zeros(T, dtype=np.int32)
    bar_count = 0
    
    # Initializing theta
    theta_plus = 0.
    theta_minus = 0.
    alpha = 2. / (span + 1)
    
    # Initializing expectations
    E_T = initial_T
    E_b_plus = initial_E_b_plus
    E_b_minus = initial_E_b_minus
    
    # initializing index of the last bar tick
    last_idx = -1
    
    # Computing min_threshold
    min_expected_plus = np.maximum(np.abs(initial_E_b_plus) * 1e-4, 1e-6)
    min_expected_minus = np.maximum(np.abs(initial_E_b_minus) * 1e-4, 1e-6)
    
    # Computing the bars indexes
    for i in range(T):
        # Computing theta plus and theta minus using simple conditions
        if tick_signs[i] > 0:
            theta_plus += tick_signs[i]
        else:
            theta_minus -= tick_signs[i]
        
        # Theta is the largest one
        theta = np.maximum(theta_plus, theta_minus)
        
        # Thresold computation
        expected_plus = np.maximum(np.abs(E_b_plus), min_expected_plus)
        expected_minus = np.maximum(np.abs(E_b_minus), min_expected_minus)
        threshold = E_T * np.maximum(expected_plus, expected_minus)
        
        if theta >= threshold and (i - last_idx) >= min_bar_length:
            # add the index
            bar_indices[bar_count] = i
            bar_count += 1
            
            # update the current mean
            current_T = i - last_idx
            current_b_plus = theta_plus / current_T
            current_b_minus = theta_minus / current_T
            
            # update the ewma
            E_T = (1 - alpha) * E_T + alpha * current_T
            E_b_plus = (1 - alpha) * E_b_plus + alpha * current_b_plus
            E_b_minus = (1 - alpha) * E_b_minus + alpha * current_b_minus
            
            # Reset
            theta_plus = 0
            theta_minus = 0
            last_idx = i
            
    # return only the non empty values
    return bar_indices[:bar_count]

In [ ]:
def tick_runs_bar_creator(ticks_dataset: pd.DataFrame,
                          ticks_first_bar: int = 1000) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["tick_rule"] = tick_rule_creator(data["price"])
    data.dropna(subset='tick_rule', inplace=True)
    
    b = data["tick_rule"].values.astype(np.int8)
    
    temp = b[:ticks_first_bar]
    initial_E_b_plus = (temp > 0).mean()
    initial_E_b_minus = 1 - initial_E_b_plus
    
    bar_indices = get_runs_bars_numba(b, 
                                      initial_T=ticks_first_bar, 
                                      initial_E_b_plus=initial_E_b_plus, 
                                      initial_E_b_minus=initial_E_b_minus)
    
    return apply_imbalance_bars(data, bar_indices)

In [ ]:
def volume_runs_bar_creator(ticks_dataset: pd.DataFrame,
                          ticks_first_bar: int = 1000) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["tick_rule"] = tick_rule_creator(data["price"])
    data.dropna(subset='tick_rule', inplace=True)
    
    bv = data["size"].values * data["tick_rule"].values.astype(np.float64)
    
    temp = bv[:ticks_first_bar]
    initial_E_b_plus = np.maximum(temp, 0.).mean()
    initial_E_b_minus = np.maximum(-temp, 0.).mean()
    
    bar_indices = get_runs_bars_numba(bv, 
                                      initial_T=ticks_first_bar, 
                                      initial_E_b_plus=initial_E_b_plus, 
                                      initial_E_b_minus=initial_E_b_minus)
    
    return apply_imbalance_bars(data, bar_indices)

In [ ]:
def dollar_runs_bar_creator(ticks_dataset: pd.DataFrame,
                          ticks_first_bar: int = 1000) -> pd.DataFrame:
    
    data = ticks_dataset[["ts_event", "price", "size"]].copy()
    
    data["tick_rule"] = tick_rule_creator(data["price"])
    data.dropna(subset='tick_rule', inplace=True)
    
    bd = (data['price'] * data["size"] * data["tick_rule"]).values.astype(np.float64)
    
    temp = bd[:ticks_first_bar]
    initial_E_b_plus = np.maximum(temp, 0.).mean()
    initial_E_b_minus = np.maximum(-temp, 0.).mean()
    
    bar_indices = get_runs_bars_numba(bd, 
                                      initial_T=ticks_first_bar, 
                                      initial_E_b_plus=initial_E_b_plus, 
                                      initial_E_b_minus=initial_E_b_minus)
    
    return apply_imbalance_bars(data, bar_indices)

In [ ]:
def plot_bar_comparison(tick_bars: pd.DataFrame, 
                        volume_bars: pd.DataFrame, 
                        dollar_bars: pd.DataFrame, 
                        start_date: str = None, 
                        end_date: str = None):
    
    # --- 0. NORMALISATION DES TIMEZONES (Le Correctif) ---
    # On crée des copies légères pour ne pas modifier vos vrais dataframes
    # On retire le fuseau horaire (.tz_localize(None)) pour que tout soit comparable
    tb = tick_bars.copy()
    vb = volume_bars.copy()
    db = dollar_bars.copy()
    
    if tb.index.tz is not None: tb.index = tb.index.tz_localize(None)
    if vb.index.tz is not None: vb.index = vb.index.tz_localize(None)
    if db.index.tz is not None: db.index = db.index.tz_localize(None)
    
    # --- 1. Filtrage temporel ---
    if start_date is None:
        # On prend une tranche au milieu des Dollar Bars
        mid = len(db) // 2
        # On s'assure de ne pas dépasser les bornes
        start_idx = max(0, mid - 500)
        end_idx = min(len(db), mid + 500)
        
        start_date = db.index[start_idx]
        end_date = db.index[end_idx]
    
    # On découpe les copies normalisées
    tb_slice = tb.loc[start_date:end_date]
    vb_slice = vb.loc[start_date:end_date]
    db_slice = db.loc[start_date:end_date]

    # --- 2. Création de la figure ---
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    
    # Graphique 1 : Tick Bars
    axes[0].plot(tb_slice.index, tb_slice['close'], color='blue', marker='.', markersize=2, linewidth=0.5)
    axes[0].set_title(f'Tick Bars (N={len(tb_slice)})', fontweight='bold')
    axes[0].set_ylabel('Prix')
    axes[0].grid(True, alpha=0.3)
    
    # Graphique 2 : Volume Bars
    axes[1].plot(vb_slice.index, vb_slice['close'], color='green', marker='.', markersize=2, linewidth=0.5)
    axes[1].set_title(f'Volume Bars (N={len(vb_slice)})', fontweight='bold')
    axes[1].set_ylabel('Prix')
    axes[1].grid(True, alpha=0.3)
    
    # Graphique 3 : Dollar Bars
    axes[2].plot(db_slice.index, db_slice['close'], color='red', marker='.', markersize=2, linewidth=0.5)
    axes[2].set_title(f'Dollar Bars (N={len(db_slice)})', fontweight='bold')
    axes[2].set_ylabel('Prix')
    axes[2].set_xlabel('Temps')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def pcaweights(
    cov: NDArray[np.floating], 
    riskDistr: Optional[NDArray[np.floating]] = None, 
    riskTarget: float = 1.) -> NDArray[np.floating]:
    
    if cov.ndim != 2 or cov.shape[0] != cov.shape[1]:
        raise ValueError("Covariance Matrix must be square")
    
    n = cov.shape[0]
    if riskDistr is None:
        riskDistr = np.zeros(n)
        riskDistr[-1] = 1.
    
    if riskDistr.ndim != 1 or riskDistr.shape[0] != n:
        raise ValueError("Risk distribution must be a vector with same dimension as covariance matrix")
    
    if not np.isclose(riskDistr.sum(), 1., atol=1e-12, rtol=0):
        raise ValueError("The risk distribution sum must be 1.")
    
    # Spectral decomposition
    d, P = np.linalg.eigh(cov)
    idx = d.argsort()[::-1]
    d, P = d[idx], P[:, idx]
    
    # Computation of beta
    beta = riskTarget * np.sqrt(riskDistr / d)
    omega = P @ beta
    
    return omega    

In [ ]:
n = 100
rng = np.random.default_rng(1)

M = rng.standard_normal((n, n))
A = M.T @ M + 1e-6 * np.eye(n)

In [ ]:
cov = A
riskDistr = np.array([1/n] * n)
omega = pcaweights(cov, riskDistr)

In [ ]:
print(omega.T @ cov @ omega)